## PROJEKT ZALICZENIOWY —— nr. 8
"Jesteś bioinformatykiem analizującym dane z sekwencjonowania pojedynczych komórek (**scRNA-seq**) pochodzących z próbek krwi obwodowej (PBMC).

Musisz zidentyfikować subpopulacje komórek odpornościowych na podstawie profili ekspresji genów. Przygotuj interaktywny raport w notatniku IPYNB (wszystkie wykresy w plotly)."

## HIPOTEZY
- **UMAP znacznie lepiej separuje subpopulacje komórkowe** niż klasyczne PCA.
- Geny markerowe lokalizują się w klastrach zgodnych z biologią.

### 1. Załadowanie bibliotek

In [70]:
import pandas as pd
import numpy as np

SEED = 42
np.random.seed(SEED)

import seaborn as sns
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "jupyterlab"
pio.templates.default = "plotly_white"

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

import scanpy as sc

COLORS = {"CD4 T cells": "#f08298", 
          "CD14+ Monocytes": "#e094a0", 
          "B cells": "#434279", 
          "CD8 T cells": "#f2b6c0", 
          "NK cells": "#5e62a9", 
          "FCGR3A+ Monocytes": "#8db7d2", 
          "Dendritic cells": "#cbc7d8", 
          "Megakaryocytes": "#c45161"}

### 2. Eksploracja danych
- pobranie danych w wersji online i offline

In [71]:
try:
    adata = sc.datasets.pbmc3k_processed()
    cell_type = 'louvain'
    print("pbmc3k_processed załadowane poprawnie :)")
except Exception as e:
    print(f'''pobieranie pbmc3k nie udało się ({type(e).__name__}).
          pbmc68k_reduced w użyciu.''')
    adata = sc.datasets.pbmc68k_reduced()
    cell_type = 'bulk_labels'

print("kształt:", adata.shape)
print("kolumny:", adata.obs.columns.tolist())
print("typy komórek:", adata.obs[cell_type].cat.categories.tolist())

pbmc3k_processed załadowane poprawnie :)
kształt: (2638, 1838)
kolumny: ['n_genes', 'percent_mito', 'n_counts', 'louvain']
typy komórek: ['CD4 T cells', 'CD14+ Monocytes', 'B cells', 'CD8 T cells', 'NK cells', 'FCGR3A+ Monocytes', 'Dendritic cells', 'Megakaryocytes']


- wypisanie liczby i typów komórek
- wypisanie liczby genów

In [72]:
n_cells_type = adata.obs[cell_type].value_counts()
print("== liczba komórek każdego typu ==")
print(n_cells_type)

n_cells = adata.obs[cell_type].value_counts().sum()
print("== liczba komórek łącznie ==")
print(n_cells)

n_genes = adata.obs["n_genes"].value_counts().sum()
print("== liczba genów ==")
print(n_genes)


== liczba komórek każdego typu ==
louvain
CD4 T cells          1144
CD14+ Monocytes       480
B cells               342
CD8 T cells           316
NK cells              154
FCGR3A+ Monocytes     150
Dendritic cells        37
Megakaryocytes         15
Name: count, dtype: int64
== liczba komórek łącznie ==
2638
== liczba genów ==
2638


- wykres słupkowy pokazujący liczbę komórek dla każdego typu komórek

In [73]:
fig = px.bar(
    x=n_cells_type.index, 
    y=n_cells_type.values,
    labels={"x": "typ komórki", "y": "liczba komórek"},
    color=n_cells_type.index, 
    color_discrete_map=COLORS, 
)
fig.update_layout(showlegend=False, width=850, height=420)
fig.update_layout(title_text="liczba komórek PBMC każdego typu", title_x=0.5)

fig.show()

### 3. PCA jako preprocessing

- PCA na macierzy ekspresji `(komórki x geny)`

In [74]:
if hasattr(adata.X, 'toarray'):
    X = adata.X.toarray()
else:
    X = np.asarray(adata.X)

labels = adata.obs[cell_type].astype(str).values

print("=== macierz ekspresji ===")
print("X kształt (shape):"               , X.shape)
print("X średnia (mean):"                , X.mean().round(3))
print("X odchylenie standardowe (std):"  , X.std().round(3))
print("ilość etykiet:"                   , len(set(labels)))

=== macierz ekspresji ===
X kształt (shape): (2638, 1838)
X średnia (mean): -0.004
X odchylenie standardowe (std): 0.931
ilość etykiet: 8


In [75]:
pca = PCA(n_components=50, random_state=SEED)
pcaX = pca.fit_transform(X)
pcaX_var_cumul = pca.explained_variance_ratio_

print("wariancja wyjaśniona przez PC1 + PC2:", round(pcaX_var_cumul.sum() * 100, 1), '%')

wariancja wyjaśniona przez PC1 + PC2: 13.0 %


- scree plot (~50 PC zawiera sygnał biologiczny)

In [76]:
pbmc_scree = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(pcaX_var_cumul))],
    "VAR": pcaX_var_cumul
})

fig = px.bar(
    pbmc_scree,
    x="PC",
    y="VAR",
    labels={"VAR": "procent wyjaśnionej wariancji",
            "PC": "główna składowa"}
)

fig.add_scatter(
    x=pbmc_scree["PC"],
    y=pbmc_scree["VAR"],
    mode="lines",
    line=dict(color="#434279", width=3),
    marker=dict(size=6),
    name="trend"
)

fig.update_traces(marker_color="#e094a0")
fig.update_layout(width=850, height=600)
fig.update_layout(title_text="scree plot dla wyników PCA", title_x=0.5)
fig.show()

- wykres PC1 oraz PC2 kolorowany typem komórki

In [77]:
df_pbmc = pd.DataFrame({
    "pca_1": pcaX[:, 0],
    "pca_2": pcaX[:, 1],
    "cell_type": labels
    })

fig = px.scatter(
    df_pbmc, x="pca_1", y="pca_2",
    color="cell_type",
    color_discrete_map=COLORS,
    labels={"pca_1": "PC1", "pca_2": "PC2"},
)
fig.update_traces(marker=dict(size=4, line=dict(width=0.2, color='white')))
fig.update_layout(width=850, height=600)
fig.update_layout(title_text="PCA na komórkach krwii, PC1 oraz PC2", title_x=0.5)
fig.show()

### 4. UMAP

- pca do 50 wymiarów

In [78]:
n_pcs = min(50, X.shape[1] - 1, X.shape[0] - 1)
pca50 = PCA(n_components=n_pcs, random_state=SEED)
pca50X = pca50.fit_transform(X)

print("kształt przed PCA50:", X.shape)
print("kształt po PCA50:", pca50X.shape)
print("wariancja wyjaśniona:", round(pca50.explained_variance_ratio_.sum() * 100, 1), '%')

kształt przed PCA50: (2638, 1838)
kształt po PCA50: (2638, 50)
wariancja wyjaśniona: 13.0 %


- UMAP dla PCA50

In [79]:
umap_mod = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=SEED,
)
umapX = umap_mod.fit_transform(pca50X)

df_pbmc["umap_1"] = umapX[:, 0]
df_pbmc["umap_2"] = umapX[:, 1]

print("kształt przed UMAP (PCA50):", pca50X.shape)
print("kształt po UMAP:", umapX.shape)

/home/oligus/miniconda3/envs/pp/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


kształt przed UMAP (PCA50): (2638, 50)
kształt po UMAP: (2638, 2)


- wykres 2D kolorowany typem komórki

In [80]:
fig = px.scatter(
    df_pbmc, x="umap_1", y="umap_2",
    color="cell_type",
    color_discrete_map=COLORS,
    labels={"umap_1": "UMAP 1", "umap_2": "UMAP 2"},
)

fig.update_traces(marker=dict(size=4, line=dict(width=0.2, color='white')))
fig.update_layout(width=850, height=600)
fig.update_layout(title_text="UMAP na PBMC, typy komórek odpornościowych", title_x=0.5)
fig.show()

### 5. Heatmapa markerów na UMAP

- wybór znanych genów markerowych

In [101]:
markers = ["NKG7", "MS4A1"]

- wykresy UMAP dla wybranych genów (kolor jest równy poziomowi ekspresji)

In [100]:
# Szukanie genów zawierających frazę "CD3" lub "CD14"
matches = [name for name in adata.var_names if "CD3D" in name or "CD3E" in name or "MS4A1" in name or "LYZ" in name or "CD14" in name]
print(f"Znalezione dopasowania: {matches}")

Znalezione dopasowania: ['MS4A1']


In [103]:
print(adata.var_names)

Index(['TNFRSF4', 'CPSF3L', 'ATAD3C', 'C1orf86', 'RER1', 'TNFRSF25', 'TNFRSF9',
       'CTNNBIP1', 'SRM', 'UBIAD1',
       ...
       'DSCR3', 'BRWD1', 'BACE2', 'SIK1', 'C21orf33', 'ICOSLG', 'SUMO3',
       'SLC19A1', 'S100B', 'PRMT2'],
      dtype='object', name='index', length=1838)
